# Арго — обучение на бесплатной видеокарте

Работает в **Google Colab** и **Kaggle**.

**Colab:** меню «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**.

**Kaggle:** справа Settings → Accelerator → **GPU T4 x2**, и включить Internet.

Запускайте ячейки по очереди сверху вниз (▶ или Shift+Enter).

## 1. Проверяем видеокарту

In [ ]:
import torch
print('Видеокарта:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ — включите GPU в настройках!')

## 2. Скачиваем код Арго

Если репозиторий закрытый (private), вместо этой ячейки загрузите файлы `model.py`, `tokenizer.py`, `train.py`, `generate.py` вручную в папку `argo_model`.

In [ ]:
!git clone -q -b claude/argo-ai-self-learning-juv44h https://github.com/pitbut/bilet.git
%cd bilet/argo/model

## 3. Загружаем книгу

Книга должна быть в формате **.txt**. Можно загрузить несколько файлов.

В **Kaggle**: загрузите книгу как Dataset (справа Add Input → Upload) и укажите путь в `BOOKS` ниже, например `/kaggle/input/moya-kniga/`.

In [ ]:
import os
os.makedirs('книги', exist_ok=True)
try:
    from google.colab import files  # Colab: появится кнопка «Выбрать файлы»
    for name, data in files.upload().items():
        open(os.path.join('книги', name), 'wb').write(data)
    BOOKS = 'книги'
except ImportError:
    BOOKS = '/kaggle/input/'  # Kaggle: поменяйте на путь к своему датасету
print('Книги:', BOOKS)

## 4. Обучаем

Каждые 250 шагов вы увидите пример текста — смотрите, как Арго из мусора учится писать.

- `--preset средний` — ~11 млн весов, хорошо для одной-нескольких книг
- `--steps` — сколько шагов (5000 ≈ 15–30 минут на T4)

In [ ]:
!python train.py --books "$BOOKS" --preset средний --steps 5000 --eval-every 250 --lr 1e-3

## 5. Говорим с Арго

In [ ]:
!python generate.py --model арго/argo.pt --prompt "Однажды" --length 600

## 6. Сохраняем модель себе

Скачайте папку `арго` (файл `argo.pt` — это и есть обученный Арго, `дневник_обучения.txt` — вся история обучения).

В Kaggle файлы из рабочей папки доступны во вкладке Output после Save Version.

In [ ]:
!zip -qr арго.zip арго
try:
    from google.colab import files
    files.download('арго.zip')
except ImportError:
    print('Kaggle: скачайте арго.zip из вкладки Output')

## 7. Дообучение на новой книге

Загрузите новую книгу (ячейка 3) и запустите — Арго продолжит учиться с того места, где остановился.

In [ ]:
!python train.py --books "$BOOKS" --resume арго/argo.pt --steps 2000 --eval-every 250 --lr 3e-4